# Настройка среды

In [ ]:
# клонируем репозиторий проекта
!git clone https://github.com/victorknox/QuittingAgents

Cloning into 'QuittingAgents'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 150 (delta 13), reused 148 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 2.88 MiB | 11.05 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [ ]:
# устанавливаем зависимости; потребуется перезапустить сеанс
%cd /content/QuittingAgents
!pip install -e .

In [ ]:
# отдельно устанавливаем все langchain модули
!pip install "langchain<1" langchain-openai langchain-anthropic langchain-community langchain-experimental

In [ ]:
# клонируем репозиторий promptcoder и устанавливаем зависимости
%cd /content/QuittingAgents
!git clone https://github.com/dhh1995/PromptCoder
%cd PromptCoder
!pip install -e .

После этого все должно работать нормально.

# Запуск эксперимента

Нам нужны ключи openai-api и openrouter-api. Добавляем их через секреты на панели слева и запускаем ячейку ниже.

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

## Отладка


In [ ]:
%cd /content/QuittingAgents
!python scripts/run.py \
  --agent-model Qwen/Qwen3-8B \
  --agent-type naive \
  --trunc-num 1 \
  --batch-size 1 \
  --auto \
  --agent-max-tokens 4096 \
  --simulator-max-tokens 2048 \
  --evaluator-max-tokens 2048 \
  --agent-max-retries 5 \
  --simulator-max-retries 5 \
  --evaluator-max-retries 5 \
  --start-index 0

## Эксперимент

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/QuittingAgents_results', exist_ok=True)

# функция backup
import shutil
from datetime import datetime

def save_results():
    timestamp = datetime.now().strftime("%m%d_%H%M%S")
    backup_path = f'/content/drive/MyDrive/QuittingAgents_results/results_{timestamp}'

    # Копируем dumps
    if os.path.exists('/content/QuittingAgents/dumps'):
        shutil.copytree('/content/QuittingAgents/dumps', backup_path)
        print(f"Results saved to Drive: {backup_path}")

    # Копируем логи
    if os.path.exists('/content/QuittingAgents/logs'):
        shutil.copytree('/content/QuittingAgents/logs',
                       f'{backup_path}/logs',
                       dirs_exist_ok=True)
    return backup_path

In [ ]:
# Конфигурации по 8 примеров за раз

configs_test = [
    (0, 8, "naive"),
    (0, 8, "simple_quit"),
    (0, 8, "quit")
]

configs_naive = [
    (0, 8, "naive"),
    (8, 8, "naive"),
    (16, 8, "naive"),
    (24, 8, "naive"),
    (32, 8, "naive"),
    (40, 8, "naive"),
    (48, 8, "naive"),
    (56, 8, "naive"),
    (64, 8, "naive")
]

configs_simple_quit = [
    (0, 8, "simple_quit"),
    (8, 8, "simple_quit"),
    (16, 8, "simple_quit"),
    (24, 8, "simple_quit"),
    (32, 8, "simple_quit"),
    (40, 8, "simple_quit"),
    (48, 8, "simple_quit"),
    (56, 8, "simple_quit"),
    (64, 8, "simple_quit")
]

configs_quit = [
    (0, 8, "quit"),
    (8, 8, "quit"),
    (16, 8, "quit"),
    (24, 8, "quit"),
    (32, 8, "quit"),
    (40, 8, "quit"),
    (48, 8, "quit"),
    (56, 8, "quit"),
    (64, 8, "quit")
]

Скрипт для эмуляции

In [ ]:
# меняем конфигурации для разных экспериментов
model = mistramistralaimistramistralai/mixtrmixtral-8x7b-instruct

for start_idx, trunc_num, agent_type in configs_simple_quit:
    # Создаем список индексов для этого батча
    selected_indices = list(range(start_idx, start_idx + trunc_num))
    indices_str = ' '.join(map(str, selected_indices))

    print(f"\n Running: {agent_type}, indices {start_idx}-{start_idx+trunc_num-1}")
    print(f"Selected indices: {indices_str}")

    %cd /content/QuittingAgents

    # Используем emulate.py напрямую с --selected-indexes
    !python scripts/emulate.py \
      --agent-model {model} \
      --agent-type {agent_type} \
      --simulator-type adv_thought \
      --selected-indexes {indices_str} \
      --batch-size 8 \
      --agent-max-tokens 4096 \
      --simulator-max-tokens 2048 \
      --agent-max-retries 5 \
      --simulator-max-retries 5 \
      --start-index {start_idx} \
      --input-path ./assets/all_cases.json \
      --trunc-num {trunc_num} \
      --track-costs \
      --dump-dir ./dumps_{agent_type}_{start_idx}

    # Сохраняем сразу после каждого запуска
    save_results()

Выходные данные были обрезаны до нескольких последних строк (5000).
    self._generate_with_cache(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/chat_models.py", line 1091, in _generate_with_cache
    result = self._generate(
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langchain_openai/chat_models/base.py", line 1213, in _generate
    raise e
  File "/usr/local/lib/python3.12/dist-packages/langchain_openai/chat_models/base.py", line 1208, in _generate
    raw_response = self.client.with_raw_response.create(**payload)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openai/_legacy_response.py", line 364, in wrapped
    return cast(LegacyAPIResponse[R], func(*args, **kwargs))
                                      ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openai/_utils/_utils.py", line 286, in wrapper
    return func(*args, **kwarg

Скрипт для оценки

In [ ]:
# После всех emulate.py запусков, добавьте оценку
import glob
import os

# Находим все файлы с результатами эмуляции
emulation_files = glob.glob("./dumps_*/*.jsonl") + glob.glob("./dumps/trajectories/*/*.jsonl")

print(f"Найдено {len(emulation_files)} файлов для оценки")

In [ ]:
# Типы оценок (safety и helpfulness)
evaluators = ['agent_safe', 'agent_help']

for emulation_file in emulation_files:
    for evaluator in evaluators:
        print(f"\n Оцениваю {os.path.basename(emulation_file)} с {evaluator}")

        !python scripts/evaluate.py \
          -inp {emulation_file} \
          -bs 8 \
          -ev {evaluator} \
          --evaluator-model-name gpt-4o-mini \
          --evaluator-max-tokens 2048 \
          --track-costs

Код для скачивания архива с данными

In [ ]:
# Архивировать папку
!zip -r dumps.zip /content/QuittingAgents

# Скачать через браузер
from google.colab import files
files.download('dumpsdumps.zip')